# Stage 0 — Data Sanity Check (rebuilt for the updated dataset)

The organizers **replaced the competition data** mid-challenge. This stage re-establishes ground truth on the new files. Key things to check now that didn't exist before:
- the data is bigger,
- the **coordinates were removed**,
- and the **test set is full of `-9999` no-data values** while training is clean.

These three facts reshape the entire approach, so we verify them first.

In [1]:
import numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# new downloaded filenames (double underscores from the browser) — do NOT rename the files
TRAIN_PATH  = "Train.csv"
TEST_PATH   = "Test.csv"
SAMPLE_PATH = "SampleSubmission.csv"
NODATA = -9999            # satellite no-data sentinel in the new test set

In [2]:
# ============================================================
# Shared feature engineering — identical for train & test.
# Handles the -9999 no-data values via NaN-aware statistics.
# ============================================================
MONTHS = [f"{i:02d}" for i in range(1, 13)]
BANDS  = ["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]

def load_raw(path):
    """Load a CSV and convert the -9999 no-data sentinel to NaN."""
    return pd.read_csv(path).replace(NODATA, np.nan)

def safe_ratio(a, b):
    # NaN propagates naturally; +0 denominator -> inf, cleaned later
    return (a - b) / (a + b)

def build_features(df):
    """Raw monthly bands -> engineered features.
    NaN-aware: every summary is computed over the VALID months only.
    Adds *_nvalid = how many months actually had data (a quality signal)."""
    feats = {}; idx = {}
    for m in MONTHS:
        g = df[f"green_{m}"]; n = df[f"nir_{m}"]; r = df[f"red_{m}"]; s1 = df[f"swir1_{m}"]
        idx.setdefault("NDWI",  []).append(safe_ratio(g,  n))
        idx.setdefault("MNDWI", []).append(safe_ratio(g,  s1))
        idx.setdefault("NDVI",  []).append(safe_ratio(n,  r))
        idx.setdefault("NDMI",  []).append(safe_ratio(n,  s1))
        idx.setdefault("VHmVV", []).append(df[f"VH_{m}"] - df[f"VV_{m}"])
    def summarize(monthly, name):
        a = np.column_stack([np.asarray(x, float) for x in monthly])  # rows x 12
        feats[f"{name}_mean"]  = np.nanmean(a, 1)
        feats[f"{name}_std"]   = np.nanstd(a, 1)
        feats[f"{name}_min"]   = np.nanmin(a, 1)
        feats[f"{name}_max"]   = np.nanmax(a, 1)
        feats[f"{name}_range"] = np.nanmax(a, 1) - np.nanmin(a, 1)
        feats[f"{name}_med"]   = np.nanmedian(a, 1)
        feats[f"{name}_nvalid"]= np.isfinite(a).sum(1)     # data-quality feature
    for b in BANDS: summarize([df[f"{b}_{m}"] for m in MONTHS], b)
    for k, v in idx.items(): summarize(v, k)
    return pd.DataFrame(feats, index=df.index).replace([np.inf, -np.inf], np.nan)

def corrupt(df, rate, seed=0):
    """Randomly blank band cells to mimic the test set's missingness."""
    rng = np.random.RandomState(seed); d = df.copy()
    bc = [c for c in d.columns if c not in ("ID", "label")]
    arr = d[bc].values.astype(float)
    arr[rng.rand(*arr.shape) < rate] = np.nan
    d[bc] = arr
    return d

In [3]:
# --- raw load (no NaN conversion yet, so we can SEE the -9999s) ---
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
sample    = pd.read_csv(SAMPLE_PATH)
print("train:", train_raw.shape, "| test:", test_raw.shape, "| sample:", sample.shape)
print("class balance:\n", train_raw["label"].value_counts(normalize=True).round(3))

train: (1821, 146) | test: (1030, 145) | sample: (1030, 3)
class balance:
 label
0    0.596
1    0.404
Name: proportion, dtype: float64


## Check 1 — are the coordinates really gone?

In [4]:
has_coords = {"lon" in train_raw.columns, "lat" in train_raw.columns}
print("lon/lat present?", has_coords)
print("non-band columns:", [c for c in train_raw.columns if not c[-3:].lstrip('_').isdigit()][:5])

lon/lat present? {False}
non-band columns: ['ID', 'label']


**Consequence:** with no `lon`/`lat`, the spatial cross-validation we relied on before is impossible. Stage 3 will replace it with **feature-space clustering**. (Coordinates were never model features anyway, so the model itself is unaffected — only the validation design is.)

## Check 2 — the no-data problem (the big one)

In [5]:
bandcols = [c for c in test_raw.columns if c not in ("ID","label")]
tr_bad = int((train_raw[bandcols]==NODATA).sum().sum())
te_bad = int((test_raw[bandcols]==NODATA).sum().sum())
print(f"train -9999 cells: {tr_bad}")
print(f"test  -9999 cells: {te_bad}  ({100*te_bad/test_raw[bandcols].size:.1f}% of all test band cells)")
rows_hit = (test_raw[bandcols]==NODATA).any(axis=1).mean()
print(f"test rows with >=1 missing cell: {100*rows_hit:.1f}%")
per_row = (test_raw[bandcols]==NODATA).sum(axis=1)
print(f"median missing cells per test row: {int(per_row.median())} of {len(bandcols)}")

train -9999 cells: 0
test  -9999 cells: 89756  (60.5% of all test band cells)
test rows with >=1 missing cell: 100.0%
median missing cells per test row: 84 of 144


**This is the heart of the new challenge.** Training is pristine; the test set has ~60% of its band values missing, and *every* test row is affected — the median location has only a handful of valid months. A pipeline that assumes clean 12-month series will collapse (we'll prove it). The whole approach now pivots on **handling missingness gracefully** and **simulating it during training** so the model isn't surprised by it.

## Check 3 — does the submission format still line up?

In [6]:
print("sample columns:", list(sample.columns))
print("test IDs carry _NEW_ tag:", test_raw["ID"].str.contains("NEW").any())
print("test IDs == sample IDs:", set(test_raw["ID"])==set(sample["ID"]))

sample columns: ['ID', 'TargetF1', 'TargetRAUC']
test IDs carry _NEW_ tag: True
test IDs == sample IDs: True


## Stage 0 takeaways (new data)
- Bigger: **1,821 train / 1,030 test**, class balance still ~40% ponds (so still no rebalancing needed).
- **Coordinates removed** → spatial CV must be replaced (Stage 3).
- **Test set ~60% missing (`-9999`), train clean** → the dominant problem; drives Stages 2, 3, 5.
- Submission format unchanged; new `_NEW_` test IDs match the sample.

**Next (Stage 1):** EDA on the new data, including how missingness is distributed and how separable the classes still are once we treat `-9999` correctly.